# 03c — Customer Behavior Analytics (Q12–Q16)

> **Session 3C: Customer segmentation and behavioral analysis**

---

## Setup

In [ ]:
# ── Connect to database ─────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from shared_setup import get_connection
conn = get_connection()

---
## Q12 — Cumulative Customer Spending

Running total spend per customer over time

In [ ]:
# Q12 — Cumulative Customer Spending
# TODO: Session 3C
customer_cumulative_spend = conn.execute('''
WITH daily_spend AS (
  SELECT
    c.customer_key,
    c.customer_id,
    d.full_date AS day,
    SUM(f.gross_amount) AS daily_spend
  FROM Fact_Order_Line f
  JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
  JOIN Dim_Date d
    ON f.date_key = d.date_key
  GROUP BY 1, 2, 3
)
SELECT
  customer_id,
  day,
  daily_spend,
  SUM(daily_spend) OVER (
    PARTITION BY customer_key
    ORDER BY day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_customer_spend
FROM daily_spend
ORDER BY customer_id, day;
''').fetchdf()

print(customer_cumulative_spend.head(50))

---
## Q13 — Inter-Purchase Time Analysis

Days between consecutive orders per customer using LAG()

In [ ]:
# Q13 — Inter-Purchase Time Analysis
# TODO: Session 3C

inter_purchase_time = conn.execute('''
WITH customer_orders AS (
  SELECT
    c.customer_key,
    c.customer_id,
    d.full_date AS order_day
  FROM Fact_Order_Line f
  JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
  JOIN Dim_Date d
    ON f.date_key = d.date_key
  GROUP BY 1, 2, 3
),
with_lag AS (
  SELECT
    customer_id,
    order_day,
    LAG(order_day) OVER (
      PARTITION BY customer_key
      ORDER BY order_day
    ) AS prev_order_day
  FROM customer_orders
)
SELECT
  customer_id,
  order_day,
  prev_order_day,
  DATE_DIFF('day', prev_order_day, order_day) AS days_since_prev_order
FROM with_lag
ORDER BY customer_id, order_day;
''').fetchdf()

print(inter_purchase_time.head(50))

---
## Q14 — Customer Recency Ranking

Rank customers by most recent purchase date

In [ ]:
# Q14 — Customer Recency Ranking
# TODO: Session 3C
customer_recency_ranking = conn.execute('''
WITH customer_last_order AS (
  SELECT
    c.customer_key,
    c.customer_id,
    MAX(d.full_date) AS last_purchase_date
  FROM Fact_Order_Line f
  JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
  JOIN Dim_Date d
    ON f.date_key = d.date_key
  GROUP BY 1, 2
)
SELECT
  customer_id,
  last_purchase_date,
  DENSE_RANK() OVER (
    ORDER BY last_purchase_date DESC
  ) AS recency_rank
FROM customer_last_order
ORDER BY recency_rank, customer_id;
''').fetchdf()

print(customer_recency_ranking.head(50))

---
## Q15 — Spending Tiers (Quartile Segmentation)

Assign customers to Bronze/Silver/Gold/Platinum using NTILE(4)

In [ ]:
# Q15 — Spending Tiers (Quartile Segmentation)
# TODO: Session 3C
spending_tiers = conn.execute('''
WITH customer_total_spend AS (
  SELECT
    c.customer_key,
    c.customer_id,
    SUM(f.gross_amount) AS total_spend
  FROM Fact_Order_Line f
  JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
  GROUP BY 1, 2
),
ranked AS (
  SELECT
    customer_id,
    total_spend,
    NTILE(4) OVER (ORDER BY total_spend DESC) AS spend_quartile
  FROM customer_total_spend
)
SELECT
  customer_id,
  total_spend,
  spend_quartile,
  CASE spend_quartile
    WHEN 4 THEN 'Platinum'
    WHEN 3 THEN 'Gold'
    WHEN 2 THEN 'Silver'
    WHEN 1 THEN 'Bronze'
  END AS spending_tier
FROM ranked
ORDER BY total_spend DESC;
''').fetchdf()

print(spending_tiers.head(50))

---
## Q16 — Top Percentile Analysis

Identify top 10% customers using PERCENT_RANK()

In [ ]:
top_10_percent_customers = conn.execute('''
WITH customer_total_spend AS (
  SELECT
    c.customer_key,
    c.customer_id,
    SUM(f.gross_amount) AS total_spend
  FROM Fact_Order_Line f
  JOIN Dim_Customer c
    ON f.customer_key = c.customer_key
  GROUP BY 1, 2
),
ranked AS (
  SELECT
    customer_id,
    total_spend,
    PERCENT_RANK() OVER (ORDER BY total_spend DESC) AS percent_rank
  FROM customer_total_spend
)
SELECT
  customer_id,
  total_spend,
  percent_rank
FROM ranked
WHERE percent_rank <= 0.10   -- top ~10% highest spenders
ORDER BY total_spend DESC;
''').fetchdf()

print(top_10_percent_customers.head(50))
  